# Libraries

In [54]:
import pandas as pd
import numpy as np

# 1. LOAD DATA

In [ ]:
accounts = pd.read_csv("datasets/ravenstack_accounts.csv")
subscriptions = pd.read_csv("datasets/ravenstack_subscriptions.csv")
usage = pd.read_csv("datasets/ravenstack_feature_usage.csv")
tickets = pd.read_csv("datasets/ravenstack_support_tickets.csv")
churn = pd.read_csv("datasets/ravenstack_churn_events_cleaned.csv")

# 2. BASIC CLEANING

In [12]:
# missing values
print(f"Missing Values\n")
print(f"Accounts\n------------\n{accounts.isnull().sum()}\n")
print(f"Subscriptions\n------------\n{subscriptions.isnull().sum()}\n")
print(f"Usage\n------------\n{usage.isnull().sum()}\n")
print(f"Tickets\n------------\n{tickets.isnull().sum()}\n")
print(f"Churn\n------------\n{churn.isnull().sum()}\n")

Missing Values

Accounts
------------
account_id         0
account_name       0
industry           0
country            0
signup_date        0
referral_source    0
plan_tier          0
seats              0
is_trial           0
churn_flag         0
dtype: int64

Subscriptions
------------
subscription_id         0
account_id              0
start_date              0
end_date             4514
plan_tier               0
seats                   0
mrr_amount              0
arr_amount              0
is_trial                0
upgrade_flag            0
downgrade_flag          0
churn_flag              0
billing_frequency       0
auto_renew_flag         0
dtype: int64

Usage
------------
usage_id               0
subscription_id        0
usage_date             0
feature_name           0
usage_count            0
usage_duration_secs    0
error_count            0
is_beta_feature        0
dtype: int64

Tickets
------------
ticket_id                        0
account_id                       0
submitted

In [20]:
# Convert dates
accounts['signup_date'] = pd.to_datetime(accounts['signup_date'])
subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'])
subscriptions['end_date'] = pd.to_datetime(subscriptions['end_date'])
usage['usage_date'] = pd.to_datetime(usage['usage_date'])
tickets['submitted_at'] = pd.to_datetime(tickets['submitted_at'])
tickets['closed_at'] = pd.to_datetime(tickets['closed_at'])
churn['churn_date'] = pd.to_datetime(churn['churn_date'])

In [21]:
# flag
subscriptions['is_active'] = subscriptions['end_date'].isna().astype(int)

In [26]:
# duplicates
print(f"Dulicates\n")
print(f"Accounts\n------------\n{accounts.duplicated().sum()}\n")
print(f"Subscriptions\n------------\n{subscriptions.duplicated().sum()}\n")
print(f"Usage\n------------\n{usage.duplicated().sum()}\n")
print(f"Tickets\n------------\n{tickets.duplicated().sum()}\n")
print(f"Churn\n------------\n{churn.duplicated().sum()}\n")

Dulicates

Accounts
------------
0

Subscriptions
------------
0

Usage
------------
0

Tickets
------------
0

Churn
------------
0



# 3. FEATURE ENGINEERING

In [28]:
# --- Subscription Aggregation ---
subs_agg = subscriptions.groupby('account_id').agg({
    'mrr_amount': 'mean',
    'arr_amount': 'mean',
    'seats': 'mean',
    'subscription_id': 'count'
}).rename(columns={
    'subscription_id': 'total_subscriptions'
}).reset_index()

# --- Usage Aggregation ---
usage_merged = pd.merge(usage, subscriptions[['subscription_id', 'account_id']], 
                        on='subscription_id', how='left')

usage_agg = usage_merged.groupby('account_id').agg({
    'usage_count': 'sum',
    'usage_duration_secs': 'sum',
    'error_count': 'sum'
}).reset_index()

# --- Support Tickets Aggregation ---
tickets_agg = tickets.groupby('account_id').agg({
    'ticket_id': 'count',
    'resolution_time_hours': 'mean',
    'first_response_time_minutes': 'mean',
    'satisfaction_score': 'mean',
    'escalation_flag': 'sum'
}).rename(columns={
    'ticket_id': 'total_tickets'
}).reset_index()

# --- Churn Flag ---
churn_flag = churn[['account_id']].copy()
churn_flag['churned'] = 1

# --- Bucket satisfaction ---

def satisfaction_bucket(x):
    if pd.isna(x):
        return "No Feedback"
    elif x >= 4:
        return "High"
    elif x >= 3:
        return "Medium"
    else:
        return "Low"

df['seat_utilization'] = df['subscription_seats'] / df['account_seats']
tickets['satisfaction_bucket'] = tickets['satisfaction_score'].apply(satisfaction_bucket)
df['tenure_days'] = (pd.Timestamp.today() - df['signup_date']).dt.days

# renaming seats columns
df.rename(columns={
    'seats_x': 'account_seats',
    'seats_y': 'subscription_seats'
}, inplace=True)

# 4. MERGE EVERYTHING

In [30]:
df = accounts.merge(subs_agg, on='account_id', how='left')
df = df.merge(usage_agg, on='account_id', how='left')
df = df.merge(tickets_agg, on='account_id', how='left')
df = df.merge(churn_flag, on='account_id', how='left')
df

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats_x,is_trial,churn_flag,...,total_subscriptions,usage_count,usage_duration_secs,error_count,total_tickets,resolution_time_hours,first_response_time_minutes,satisfaction_score,escalation_flag,churned
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False,...,10,535,152339,38,2.0,23.000000,91.000000,3.000000,0.0,1.0
1,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False,...,10,535,152339,38,2.0,23.000000,91.000000,3.000000,0.0,1.0
2,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True,...,8,355,101136,14,3.0,38.000000,73.333333,4.000000,0.0,NaN
3,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False,...,15,821,251210,48,3.0,43.666667,63.666667,4.666667,0.0,1.0
4,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False,...,15,821,251210,48,3.0,43.666667,63.666667,4.666667,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
743,A-55f257,Company_496,FinTech,US,2023-12-21,organic,Basic,9,False,False,...,10,476,145972,25,6.0,43.166667,114.500000,5.000000,0.0,NaN
744,A-d26ab4,Company_497,DevTools,UK,2024-11-07,organic,Basic,9,False,True,...,9,458,121379,24,2.0,49.500000,110.000000,4.000000,0.0,1.0
745,A-712533,Company_498,EdTech,US,2023-07-31,organic,Pro,18,False,False,...,12,571,180056,35,5.0,37.400000,97.200000,3.750000,0.0,1.0
746,A-781cc0,Company_499,EdTech,US,2024-09-04,organic,Pro,12,False,False,...,16,864,270008,42,4.0,57.500000,106.000000,3.666667,0.0,1.0


# Fill missing values

In [31]:
df.fillna(0, inplace=True)

# 5. FINAL DATASET CHECK

In [37]:
print(f"Shape \n--------\n{df.shape}\n--------\n")
print(df.isnull().sum())

Shape 
--------
(748, 23)
--------

account_id                     0
account_name                   0
industry                       0
country                        0
signup_date                    0
referral_source                0
plan_tier                      0
seats_x                        0
is_trial                       0
churn_flag                     0
mrr_amount                     0
arr_amount                     0
seats_y                        0
total_subscriptions            0
usage_count                    0
usage_duration_secs            0
error_count                    0
total_tickets                  0
resolution_time_hours          0
first_response_time_minutes    0
satisfaction_score             0
escalation_flag                0
churned                        0
dtype: int64


In [50]:
df.columns

Index(['account_id', 'account_name', 'industry', 'country', 'signup_date',
       'referral_source', 'plan_tier', 'account_seats', 'is_trial',
       'churn_flag', 'mrr_amount', 'arr_amount', 'subscription_seats',
       'total_subscriptions', 'usage_count', 'usage_duration_secs',
       'error_count', 'total_tickets', 'resolution_time_hours',
       'first_response_time_minutes', 'satisfaction_score', 'escalation_flag',
       'churned', 'tenure_days', 'seat_utilization'],
      dtype='object')

In [52]:
df.head()

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,account_seats,is_trial,churn_flag,...,usage_duration_secs,error_count,total_tickets,resolution_time_hours,first_response_time_minutes,satisfaction_score,escalation_flag,churned,tenure_days,seat_utilization
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False,...,152339,38,2.0,23.000000,91.000000,3.000000,0.0,1.0,550,3.466667
1,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False,...,152339,38,2.0,23.000000,91.000000,3.000000,0.0,1.0,550,3.466667
2,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True,...,101136,14,3.0,38.000000,73.333333,4.000000,0.0,0.0,976,1.222222
3,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False,...,251210,48,3.0,43.666667,63.666667,4.666667,0.0,1.0,600,18.800000
4,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False,...,251210,48,3.0,43.666667,63.666667,4.666667,0.0,1.0,600,18.800000


In [53]:
# save cleaned file
df.to_csv("saas_cleaned_data.csv", index=False)
print("Saved cleaned dataset")

Saved cleaned dataset
